In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

# 데이터 로드 및 변환
timesteps = 13
features = 2  # 🔹 Feature 개수: 원래 데이터 + 평균값

# CSV 데이터 불러오기
punch_data = pd.read_csv("punch.csv", usecols=range(13))
no_punch_data = pd.read_csv("normal.csv", usecols=range(13))
pet_data = pd.read_csv("pet.csv", usecols=range(13))
pinch_data = pd.read_csv("pinch.csv", usecols=range(13))

# NumPy 변환 후 float32 형 변환
punch_data = np.array(punch_data).astype(np.float32)
no_punch_data = np.array(no_punch_data).astype(np.float32)
pet_data = np.array(pet_data).astype(np.float32)
pinch_data = np.array(pinch_data).astype(np.float32)

# 샘플 개수
samples_punch = len(punch_data)
samples_no_punch = len(no_punch_data)
samples_pet = len(pet_data)
samples_pinch = len(pinch_data)

# 🔹 데이터 합치기
X_data = np.concatenate([punch_data, no_punch_data, pet_data, pinch_data], axis=0)  # (총 샘플 수, 13)

# 🔹 평균값을 추가 (샘플별 평균 계산 후, 모든 타임스텝에 추가)
row_means = np.mean(X_data, axis=1, keepdims=True)  # (샘플 수, 1)
row_means_expanded = np.tile(row_means, (1, timesteps)).reshape(-1, timesteps, 1)  # (샘플 수, 13, 1)

# 🔹 기존 데이터 차원 확장 후 평균값 추가 (Feature 개수: 2)
X_data = np.expand_dims(X_data, axis=-1)  # (샘플 수, 13, 1)
X_data = np.concatenate([X_data, row_means_expanded], axis=-1)  # (샘플 수, 13, 2)

# 🔹 레이블 생성 (0: 펀치, 1: 비펀치, 2: 애완동물, 3: 핀치)
y_data = np.concatenate([
    np.zeros((samples_punch, 1)),  # 0: 펀치
    np.ones((samples_no_punch, 1)),  # 1: 비펀치
    np.full((samples_pet, 1), 2),  # 2: 애완동물
    np.full((samples_pinch, 1), 3)  # 3: 핀치
], axis=0)

# PyTorch Tensor 변환
X_data = torch.tensor(X_data, dtype=torch.float32)
y_data = torch.tensor(y_data, dtype=torch.long).squeeze()

# 데이터셋 나누기
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=42)

# DataLoader 사용
batch_size = 4
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

#  LSTM 모델 정의 (🔹 input_size=2로 변경)
class PunchDetectionLSTM(nn.Module):
    def __init__(self, num_classes=4):
        super(PunchDetectionLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=2, hidden_size=64, num_layers=2, batch_first=True)  # 🔹 input_size=2
        self.fc1 = nn.Linear(64, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, num_classes)  # 🔹 4개 클래스 (펀치, 비펀치, 애완동물, 핀치)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        x = self.fc1(lstm_out[:, -1, :])  # 마지막 타임스텝 사용
        x = self.relu(x)
        x = self.fc2(x)
        return x

#  모델 학습
model = PunchDetectionLSTM()
criterion = nn.CrossEntropyLoss()  # 🔹 다중 분류를 위한 CrossEntropyLoss
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20
for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # 검증 데이터 평가
    model.eval()
    val_loss = 0.0
    correct, total = 0, 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()

            # 🔹 정확도 계산
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    accuracy = 100 * correct / total
    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss / len(train_loader):.4f}, "
          f"Val Loss: {val_loss / len(test_loader):.4f}, Accuracy: {accuracy:.2f}%")

# 모델 저장
torch.save(model.state_dict(), "please.pth")
print(" 모델이 저장되었습니다!")


Epoch [1/20], Train Loss: 0.7028, Val Loss: 0.3519, Accuracy: 77.48%
Epoch [2/20], Train Loss: 0.2925, Val Loss: 0.5673, Accuracy: 77.48%
Epoch [3/20], Train Loss: 0.3107, Val Loss: 0.3465, Accuracy: 77.48%
Epoch [4/20], Train Loss: 0.2689, Val Loss: 0.2068, Accuracy: 94.04%
Epoch [5/20], Train Loss: 0.1377, Val Loss: 0.0891, Accuracy: 97.35%
Epoch [6/20], Train Loss: 0.1059, Val Loss: 0.1909, Accuracy: 94.70%
Epoch [7/20], Train Loss: 0.1009, Val Loss: 0.1608, Accuracy: 90.73%
Epoch [8/20], Train Loss: 0.0906, Val Loss: 0.1075, Accuracy: 96.69%
Epoch [9/20], Train Loss: 0.0603, Val Loss: 0.0789, Accuracy: 97.35%
Epoch [10/20], Train Loss: 0.1155, Val Loss: 0.1037, Accuracy: 96.69%
Epoch [11/20], Train Loss: 0.0774, Val Loss: 0.1430, Accuracy: 93.38%
Epoch [12/20], Train Loss: 0.1096, Val Loss: 0.1862, Accuracy: 94.70%
Epoch [13/20], Train Loss: 0.0794, Val Loss: 0.0855, Accuracy: 98.01%
Epoch [14/20], Train Loss: 0.0715, Val Loss: 0.0791, Accuracy: 98.01%
Epoch [15/20], Train Loss: 0.

In [5]:
import torch
import torch.nn as nn
import numpy as np
import serial

# 🔹 시리얼 포트 설정
SERIAL_PORT = "COM11"  # 사용 중인 포트로 변경하세요
BAUD_RATE = 9600

# 🔹 클래스 라벨 정의
class_labels = {0: "펀치", 1: "노말", 2: "쓰다듬기", 3: "꼬집기"}

# 🔹 저장된 LSTM 모델 불러오기
class PunchDetectionLSTM(nn.Module):
    def __init__(self, num_classes=4):
        super(PunchDetectionLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=2, hidden_size=64, num_layers=2, batch_first=True)
        self.fc1 = nn.Linear(64, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, num_classes)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        x = self.fc1(lstm_out[:, -1, :])  # 마지막 타임스텝 사용
        x = self.relu(x)
        x = self.fc2(x)
        return x

# 🔹 모델 초기화 및 가중치 로드
model = PunchDetectionLSTM()
model.load_state_dict(torch.load("please.pth"))
model.eval()  # 평가 모드

def process_lstm_model(values):
    """
    LSTM 모델을 사용하여 실시간 데이터 분류
    """
    # 평균값 계산 후 feature 추가
    avg = np.mean(values)
    new_data = np.array(values).reshape(1, 13, 1)  # (1, 13, 1)
    avg_feature = np.full((1, 13, 1), avg)  # 평균값을 모든 타임스텝에 추가 (1, 13, 1)

    # 🔹 입력 데이터 변환 (샘플 1개, 타임스텝 13, Feature 2개)
    input_data = np.concatenate([new_data, avg_feature], axis=-1)  # (1, 13, 2)
    input_tensor = torch.tensor(input_data, dtype=torch.float32)

    # 🔹 모델 예측
    with torch.no_grad():
        outputs = model(input_tensor)
        predicted_class = torch.argmax(outputs, dim=1).item()

    print(f"예측 결과: {class_labels[predicted_class]}\n")

# 🔹 시리얼 데이터 수신
try:
    ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
    print(f"시리얼 포트 {SERIAL_PORT} 연결됨!")

    while True:
        line = ser.readline().decode('utf-8').strip().rstrip(",")  # 시리얼 데이터 읽기

        if line:
            try:
                values = [float(x) for x in line.split(',')]  # 숫자로 변환

                if len(values) == 13:
                    process_lstm_model(values)  # LSTM 모델로 예측 수행
                else:
                    print(f"데이터 길이 오류 (13개 필요, 현재: {len(values)}) → {values}")

            except ValueError:
                print(f"변환 오류: {line}")

except serial.SerialException as e:
    print(f"시리얼 포트 연결 실패: {e}")
except KeyboardInterrupt:
    print("프로그램 종료")
finally:
    if 'ser' in locals() and ser.is_open:
        ser.close()
        print(f"시리얼 포트 {SERIAL_PORT} 닫힘")


시리얼 포트 COM11 연결됨!
데이터 길이 오류 (13개 필요, 현재: 6) → [15.1, 7.22, 13.27, 26.49, 30.73, 22.28]
예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 펀치

예측 결과: 펀치

예측 결과: 노말

예측 결과: 쓰다듬기

예측 결과: 쓰다듬기

예측 결과: 노말

예측 결과: 펀치

예측 결과: 쓰다듬기

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 쓰다듬기

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 꼬집기

예측 결과: 꼬집기

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 펀치

예측 결과: 노말

예측 결과: 노말

예측 결과: 펀치

예측 결과: 쓰다듬기

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 쓰다듬기

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 펀치

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 쓰다듬기

예측 결과: 쓰다듬기

예측 결과: 쓰다듬기

예측